In [1]:
import os, json, time
from pathlib import Path
import numpy as np, pandas as pd
from openai import OpenAI
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier)
from sklearn.metrics import roc_auc_score
from scipy.stats import wilcoxon
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

OUT_DIR = Path("./nsduh_analysis_outputs")
DATA_PATH = OUT_DIR / "df_corrected_7970_with_gpt_profiles_embeddings.csv"
DIRECT_CACHE = OUT_DIR / "direct_text_embeddings_7970.csv"
EMBED_MODEL = "text-embedding-3-small"


if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set the OPENAI_API_KEY environment variable before running this notebook.")

In [2]:
# Load
df = pd.read_csv(DATA_PATH)
y = df["cost_barrier"].astype(int).values


# Feature 1: Raw Structured (00노트북과 동일)
raw_cols = ["year", "AGE3", "NEWRACE2", "IRINSUR4", "substance_peer_support", "mental_health_peer_support"]
X_raw = pd.get_dummies(df[raw_cols].astype(str), drop_first=False).values.astype("float32")


# Feature 2: GPT profile embedding
X_gpt = np.vstack(df["predictive_embedding"].apply(json.loads).values).astype("float32")

In [3]:
# Feature 3: Direct text embedding (신규, 캐시)
def get_embeddings_batch(client, texts, max_retries=5):
    clean = ["" if pd.isna(t) else str(t) for t in texts]
    for attempt in range(max_retries):
        try:
            resp = client.embeddings.create(model=EMBED_MODEL, input=clean, timeout=120)
            return [d.embedding for d in resp.data]
            # return
        except Exception as e:
            print(f"[WARN] embed attempt {attempt+1}: {e}")
            time.sleep(min(2**attempt, 60))
    return [None]*len(texts)

if DIRECT_CACHE.exists():
    print("[INFO] loading cached direct embeddings")
    dfe = pd.read_csv(DIRECT_CACHE)
    assert (dfe["original_index"].values == df["original_index"].values).all(), "row order mismatch"
    X_direct = np.vstack(dfe["direct_embedding"].apply(json.loads).values).astype("float32")
else:
    print("[INFO] generating direct embeddings via API")
    client = OpenAI(timeout=120.0, max_retries=2)
    texts = df["predictive_text"].tolist()
    embs, BATCH = [], 256
    for i in range(0, len(texts), BATCH):
        embs.extend(get_embeddings_batch(client, texts[i:i+BATCH]))
        print(f" embedded {min(i+BATCH, len(texts))}/{len(texts)}")
    assert all(e is not None for e in embs), "some ebedidngs failed"
    pd.DataFrame({"original_index": df["original_index"].values,
                 "direct_embedding":[json.dumps(e) for e in embs]}).to_csv(DIRECT_CACHE, index=False)
    X_direct = np.vstack([np.asarray(e, dtype="float32") for e in embs])

print("shapes:", X_raw.shape, X_direct.shape, X_gpt.shape)

[INFO] loading cached direct embeddings
shapes: (7970, 26) (7970, 1536) (7970, 1536)


In [4]:
# 돌인 고정 split

idx = np.arange(len(df))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=y)
yr_test = df["year"].values[test_idx]

feature_sets = {
    "Raw structured":        X_raw,
    "Direct text embedding": X_direct,
    "GPT profile embedding": X_gpt,
}

def make_model(name):  # 00 노트북 cell 40 그대로
    if name == "Logistic Regression":
        return make_pipeline(StandardScaler(),
            LogisticRegression(max_iter=3000, solver="saga",
                               class_weight="balanced", n_jobs=-1, random_state=42))
    if name == "Linear SVM":
        return make_pipeline(StandardScaler(),
            LinearSVC(class_weight="balanced", max_iter=5000, random_state=42))
    if name == "Ridge Classifier":
        return make_pipeline(StandardScaler(),
            RidgeClassifier(class_weight="balanced", random_state=42))
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                      n_jobs=-1, class_weight="balanced", random_state=42)
    if name == "Extra Trees":
        return ExtraTreesClassifier(n_estimators=300, min_samples_leaf=5,
                                    n_jobs=-1, class_weight="balanced", random_state=42)
    if name == "HistGradientBoosting":
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                              max_leaf_nodes=31, l2_regularization=0.01,
                                              random_state=42)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                             subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
                             n_jobs=-1, random_state=42)
    raise ValueError(name)

MODELS = ["Logistic Regression","Linear SVM","Ridge Classifier","Random Forest",
          "Extra Trees","HistGradientBoosting"] + (["XGBoost"] if HAS_XGB else [])

def get_score(model, X):
    if hasattr(model, "predict_proba"):    return model.predict_proba(X)[:,1]
    if hasattr(model, "decision_function"): return model.decision_function(X)
    return model.predict(X)

rows = []
for m in MODELS:
    for fname, X in feature_sets.items():
        model = make_model(m); model.fit(X[train_idx], y[train_idx])
        s = get_score(model, X[test_idx])
        m22, m23 = yr_test==2022, yr_test==2023
        rows.append(dict(model=m, feature=fname,
                         auc=roc_auc_score(y[test_idx], s),
                         auc_2022=roc_auc_score(y[test_idx][m22], s[m22]),
                         auc_2023=roc_auc_score(y[test_idx][m23], s[m23])))
res = pd.DataFrame(rows)
res.to_csv(OUT_DIR/"direct_embedding_control_results.csv", index=False)

/data/users/jupyter-yos225/venvs/proflee/drug/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/data/users/jupyter-yos225/venvs/proflee/drug/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/data/users/jupyter-yos225/venvs/proflee/drug/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/data/users/jupyter-yos225/venvs/proflee/drug/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided '

In [8]:
# pivot + 분류기 간 Wilcoxon
pivot = res.pivot(index="model", columns="feature", values="auc")
def wilx(a, b):
    d = (pivot[a]-pivot[b]).values
    stat, p = wilcoxon(d, alternative="greater")
    return dict(comparison=f"{a} - {b}", n=len(d), mean_diff=d.mean(),
                median_diff=float(np.median(d)), n_pos=int((d>0).sum()),
                wilcoxon_stat=stat, p_one_sided=p)
comp = pd.DataFrame([wilx("Direct text embedding","Raw structured"),
                     wilx("GPT profile embedding","Raw structured"),   # sanity: 0.0391 재현 기대
                     wilx("GPT profile embedding","Direct text embedding")])
comp.to_csv(OUT_DIR/"direct_embedding_control_wilcoxon.csv", index=False)

pd.set_option("display.width", 140)
print("\n=== per-model AUC (all / 2022 / 2023) ===\n", res.to_string(index=False))
print("\n=== pivot (overall AUC) ===\n", pivot.round(4).to_string())
print("\n=== Wilcoxon across classifiers ===\n", comp.to_string(index=False))


=== per-model AUC (all / 2022 / 2023) ===
                model               feature      auc  auc_2022  auc_2023
 Logistic Regression        Raw structured 0.687130  0.678890  0.693050
 Logistic Regression Direct text embedding 0.666455  0.643395  0.693977
 Logistic Regression GPT profile embedding 0.697810  0.695143  0.701943
          Linear SVM        Raw structured 0.686547  0.678626  0.692168
          Linear SVM Direct text embedding 0.662854  0.635595  0.691560
          Linear SVM GPT profile embedding 0.685488  0.680885  0.690474
    Ridge Classifier        Raw structured 0.686527  0.678626  0.692168
    Ridge Classifier Direct text embedding 0.664247  0.637005  0.693451
    Ridge Classifier GPT profile embedding 0.686306  0.682319  0.691295
       Random Forest        Raw structured 0.678975  0.666076  0.694601
       Random Forest Direct text embedding 0.661725  0.640062  0.688238
       Random Forest GPT profile embedding 0.690550  0.682646  0.700915
         Extra Trees

In [9]:
# Table 6: 연도별 (GPT-Raw), (Direct-Raw) gap 비교
res = pd.read_csv(OUT_DIR / "direct_embedding_control_results.csv")

wide = res.pivot(index="model", columns="feature", values=["auc_2022", "auc_2023"])
wide.columns = [f"{feat}_{yr.split('_')[1]}" for yr, feat in wide.columns]

gap = pd.DataFrame(index=wide.index)
gap["GPT-Raw gap (2022)"]    = wide["GPT profile embedding_2022"] - wide["Raw structured_2022"]
gap["GPT-Raw gap (2023)"]    = wide["GPT profile embedding_2023"] - wide["Raw structured_2023"]
gap["Direct-Raw gap (2022)"] = wide["Direct text embedding_2022"] - wide["Raw structured_2022"]
gap["Direct-Raw gap (2023)"] = wide["Direct text embedding_2023"] - wide["Raw structured_2023"]
gap = gap.round(4)

gap.to_csv(OUT_DIR / "year_split_gap_table.csv")
print(gap.to_string())
print("\nMean across 7 classifiers:")
print(gap.mean().round(4))

                      GPT-Raw gap (2022)  GPT-Raw gap (2023)  Direct-Raw gap (2022)  Direct-Raw gap (2023)
model                                                                                                     
Extra Trees                       0.0185              0.0118                -0.0288                -0.0036
HistGradientBoosting              0.0256              0.0031                -0.0242                -0.0028
Linear SVM                        0.0023             -0.0017                -0.0430                -0.0006
Logistic Regression               0.0163              0.0089                -0.0355                 0.0009
Random Forest                     0.0166              0.0063                -0.0260                -0.0064
Ridge Classifier                  0.0037             -0.0009                -0.0416                 0.0013
XGBoost                           0.0210              0.0215                -0.0353                -0.0009

Mean across 7 classifiers:
GPT-Raw g

In [10]:
res = pd.read_csv(OUT_DIR / "direct_embedding_control_results.csv")
table_6a = res.pivot(index="model", columns="feature", values="auc").round(4)
table_6a = table_6a[["Raw structured", "Direct text embedding", "GPT profile embedding"]]
print(table_6a.to_string())

feature               Raw structured  Direct text embedding  GPT profile embedding
model                                                                             
Extra Trees                   0.6763                 0.6606                 0.6910
HistGradientBoosting          0.6718                 0.6584                 0.6862
Linear SVM                    0.6865                 0.6629                 0.6855
Logistic Regression           0.6871                 0.6665                 0.6978
Random Forest                 0.6790                 0.6617                 0.6905
Ridge Classifier              0.6865                 0.6642                 0.6863
XGBoost                       0.6863                 0.6664                 0.7052
